#  Incident Response: Tier 0 Domain Compromise (Pirate Hunt)

## 1. Executive Summary
Our threat hunting team detected an advanced Active Directory compromise targeting our Tier 0 infrastructure (Domain Controllers). The adversary executed a multi-stage attack leveraging legacy group permissions, NTLM Coercion, and ultimately **Kerberos Constrained Delegation (KCD) Hijacking** via **Service Principal Name (SPN) Manipulation**.

### Hunt Hypothesis & Behavioral Outliers
* **Hypothesis:** Adversaries bypassing standard EDR will generate anomalous directory replication or modification events, specifically targeting delegation attributes and SPNs originating from non-system endpoints.
* **Outliers Detected:** A delegated user account (`a.white_adm`) was observed removing and immediately reallocating a `servicePrincipalName` across two distinct machine accounts (`WEB01$` to `DC01$`), indicating an active KCD hijacking attempt.
* **Toxic Combinations Exploited:** `Pre-Windows 2000 Compatible Access` + Accounts with `WriteSPN` + Kerberos Constrained Delegation.

## 2. Summary of Attack Chain
| Step | User / Access         | Technique Used                            | Result                                                                                                         |
| :--: | :-------------------- | :---------------------------------------- | :------------------------------------------------------------------------------------------------------------- |
|   1  | N/A (External)        | **Active Directory Enumeration**          | Identified `MS01$` as a member of **Pre-Windows 2000 Compatible Access** via `adscan` and `bloodhound-python`. |
|   2  | MS01$                 | **AS-REQ (Default Machine Password)**     | Requested a TGT for `MS01$` using the machine name (`ms01`) as its password.                                   |
|   3  | MS01$                 | **LDAP Read (gMSA Extraction)**           | Dumped `gMSA_ADFS_prod$` NT hash by reading `msDS-ManagedPassword` through legacy group privileges.            |
|   4  | gMSA_ADFS_prod$       | **Pass-the-Hash (WinRM)**                 | Authenticated to `DC01` using **Evil-WinRM**, establishing initial foothold.                                   |
|   5  | gMSA_ADFS_prod$       | **Network Pivoting**                      | Created Layer-3 tunnel to internal subnet `192.168.100.0/24` using **Ligolo-ng**.                              |
|   6  | gMSA_ADFS_prod$       | **NTLM Coercion & Relaying**              | Coerced `WEB01$` authentication and relayed NTLM to `DC01` over LDAPS to perform RBCD attack.                  |
|   7  | gMSA_ADFS_prod$       | **RBCD & S4U Impersonation**              | Injected rogue computer `VYSHKGDW$` and forged service ticket for `Administrator` on `WEB01`.                  |
|   8  | Administrator (WEB01) | **Lateral Movement (User Flag)**          | Used **Impacket** `psexec.py` to gain admin shell on `WEB01` and retrieve **user.txt**.                        |
|   9  | Administrator (WEB01) | **Credential Harvesting**                 | Dumped local secrets to recover plaintext password for `a.white`.                                              |
|  10  | a.white               | **ACL Tiering Violation**                 | Abused `ForceChangePassword` rights to overwrite password for `a.white_adm`.                                   |
|  11  | a.white_adm           | **SPN Injection**                         | Removed `HTTP` SPN from `WEB01$` and injected into `DC01$`, hijacking constrained delegation path.             |
|  12  | a.white_adm           | **Kerberos Constrained Delegation (S4U)** | Requested forged `CIFS` ticket to `DC01` impersonating Domain Admin.                                           |
|  13  | Administrator (DC01)  | **Pass-the-Ticket (Root Flag)**           | Used forged ticket with `psexec.py` to access `DC01` and retrieve **root.txt**.                                |


In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner presentation
import warnings
warnings.filterwarnings('ignore')

print("[*] Initializing Forensic Environment & Loading Plaso Telemetry...")

# Load the dataset
df = pd.read_csv('detections_data.csv', low_memory=False)

# Extract Event IDs robustly from the Plaso message string
df['event_id'] = pd.to_numeric(df['message'].str.extract(r'^\[(\d+)\s*/')[0], errors='coerce').fillna(0).astype(int)

print(f"[+] Successfully loaded {len(df)} total events.")

[*] Initializing Forensic Environment & Loading Plaso Telemetry...
[+] Successfully loaded 65663 total events.


## 3. Forensic Evidence: Initial Access & NTLM Coercion (Steps 4 - 6)
To prove the beginning of the attack chain, we hunted for the compromised Group Managed Service Account (`gMSA_ADFS_prod$`). The logs revealed anomalous WinRM tunneling from an external pivot IP (`192.168.100.2`), followed immediately by rogue DNS node creation, setting the stage for NTLM coercion.

In [2]:
print("[*] Correlating Initial Access & DNS Spoofing Indicators...")

# 1. Hunt for Step 4 & 5: gMSA_ADFS_prod$ Anomalous WinRM Connections (Event 2947)
gmsa_auth = df[(df['event_id'] == 2947) & (df['message'].str.contains('gMSA_ADFS_prod', case=False, na=False))].copy()

if not gmsa_auth.empty:
    gmsa_auth['Attacker_Account'] = gmsa_auth['message'].str.extract(r'CN=(gMSA_ADFS_prod)')
    gmsa_auth['Source_IP_Port'] = gmsa_auth['message'].str.extract(r'(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}:\d+)')
    
    print("\n[!] EVIDENCE (Step 4 & 5): Pass-the-Hash & WinRM Tunnel Detected")
    display(gmsa_auth[['datetime', 'event_id', 'Attacker_Account', 'Source_IP_Port']].head(1).style.set_properties(**{'background-color': '#fff3f3', 'border-color': 'gray'}))

# 2. Hunt for Step 6: Rogue DNS Node Creation for NTLM Coercion (Event 5136)
gmsa_dns = df[(df['event_id'] == 5136) & 
              (df['message'].str.contains('gMSA_ADFS_prod', na=False, case=False)) &
              (df['message'].str.contains('dnsNode', na=False, case=False))].copy()

if not gmsa_dns.empty:
    gmsa_dns['Attacker_Account'] = gmsa_dns['message'].str.extract(r'Account Name:\\t\\t(.*?)\\n')
    gmsa_dns['Rogue_DNS_Node'] = gmsa_dns['message'].str.extract(r'DN:\\t(.*?)\\n')
    gmsa_dns['Action'] = "DNS Spoofing (NTLM Coercion Setup)"
    
    print("\n[!] EVIDENCE (Step 6): Rogue DNS Injection for NTLM Relaying Detected")
    display(gmsa_dns[['datetime', 'Attacker_Account', 'Action', 'Rogue_DNS_Node']].head(2).style.set_properties(**{'background-color': '#fff3f3', 'border-color': 'gray'}))

[*] Correlating Initial Access & DNS Spoofing Indicators...

[!] EVIDENCE (Step 4 & 5): Pass-the-Hash & WinRM Tunnel Detected


,datetime,event_id,Attacker_Account,Source_IP_Port
8025,2025-06-09T17:48:41.954478+00:00,2947,gMSA_ADFS_prod,192.168.100.2:49769



[!] EVIDENCE (Step 6): Rogue DNS Injection for NTLM Relaying Detected


,datetime,Attacker_Account,Action,Rogue_DNS_Node
10311,2025-06-11T14:09:27.132715+00:00,gMSA_ADFS_prod$,DNS Spoofing (NTLM Coercion Setup),DC=test1 DC=pirate.htb CN=MicrosoftDNS DC=DomainDnsZones DC=pirate DC=htb
10313,2025-06-11T14:09:34.790824+00:00,gMSA_ADFS_prod$,DNS Spoofing (NTLM Coercion Setup),DC=test1 DC=pirate.htb CN=MicrosoftDNS DC=DomainDnsZones DC=pirate DC=htb


## 4. Forensic Evidence: KCD SPN Hijacking (Step 11)
After exploiting Tiering violations to compromise `a.white_adm`, the adversary executed the final phase: moving the `HTTP` SPN from the targeted Web Server to the Domain Controller. This is the exact moment Tier 0 was compromised.